# Simple Anomaly Detection Analysis

This notebook is organized for paper-facing experiments rather than one-off benchmarking.

Main ideas:

- All cross-cell state stays inside `NOTEBOOK_STATE`.
- The paper presets create multiple named variants per algorithm so the notebook can show parameter effects instead of a single baseline.
- `Argument mode` can now switch between manual subtabs and automatic preset sweeps.
- `auto_ablation` creates one baseline plus one-knob-at-a-time variants so each parameter claim is paired against a fixed reference.
- Paper-facing sweeps vary only score-driving parameters; backend-only or threshold-only knobs stay fixed inside the presets.
- Benchmark outputs are saved into `results/tables/`, `results/figures/`, and `results/scores/`.
- The notebook can export a fixed-style thesis figure pack with matching caption text in one step.

Default workflow:

1. Run the dependency and setup cells.
2. Optionally apply `paper_high_roi`, `paper_full_suite`, or `auto_ablation`.
3. Run the preparation and benchmark cells.
4. Inspect the regime summary and winner tables.
5. Open the per-algorithm report cells for the paper figures.
6. Run the thesis export cell to write PNG/PDF figures and captions.

## Control Reference

**General**
- `Argument mode`: `manual` uses the current subtabs, while the auto modes expand enabled algorithms into preset parameter combinations.
- `Dataset limit`: how many prepared datasets to benchmark. `0` means all datasets.
- `Normalize`: preprocessing applied before any algorithm runs.
- `Clip q`: optional quantile clipping before normalization.
- `Window size`: base temporal context length for subsequence-aware processing. `0` keeps automatic estimation.
- `Window stride`: step between consecutive windows or subsequences. Larger strides reduce overlap and runtime.
- `Threshold method` with `Threshold value`: turns continuous scores into binary detections for the paper metrics.
- `Evaluation mode`: `range` emphasizes overlap with anomaly intervals, while `point` is stricter and judges exact indices.
- `Rebuild normalized datasets`: regenerates cached prepared datasets.
- `Save per-dataset scores`: writes raw score traces to `results/scores/`.

**Paper sweep rules**
- `paper_high_roi`: short, paper-friendly sweep on the highest-return methods plus Isolation Forest for calibration discussion.
- `paper_full_suite`: broader appendix-ready sweep across every implemented method.
- `auto_ablation`: one baseline plus one-knob-at-a-time variants, intended for defensible sensitivity claims and presentation-ready parameter analysis.
- In auto modes, the notebook ignores the live subtab values at run time and uses the preset variants instead.
- Presets keep runtime-only knobs fixed and vary only parameters that materially change each algorithm's scoring behavior in this framework.

**Algorithm sweep highlights**
- Isolation Forest: `Trees`, `Max samples`, `Max feat.`, `Bootstrap`.
- Local Outlier Factor: `Neighbors`, `Metric`, `p`.
- SAND: `Alpha`, `Init length`, `Batch size`, `k`, `Subseq x`, `Overlap`.
- Matrix Profile: `Subseq x`.
- DAMP: `Start x`, `x_lag x`.
- HBOS: `Bins`, `Alpha`, `Tol`.
- OCSVM: `Kernel`, `Nu`, `Gamma`, `Train frac`.
- PCA: `Components`, `Score comps`, `Whiten`, `Weighted`, `Standardize`.

## On Run: Check Whether Notebook Dependencies Are Installed And Install Any Missing Ones

This next cell checks for `ipywidgets`, `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `stumpy`, and `TSB-UAD`, then installs any missing package into the current Python environment.

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "ipywidgets": "ipywidgets",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "stumpy": "stumpy",
    "TSB_UAD": "TSB-UAD",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Notebook dependencies are available.")

Notebook dependencies are available.


## On Run: Locate The Project, Reload Notebook Support, And Render The Interactive Control Panel

This next cell finds the `simple_anomaly_detection` folder, reloads the support module, loads dataset names, renders the control panel, writes the algorithm-mechanics notes, and initializes `NOTEBOOK_STATE`.

In [ ]:
import importlib
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

search_bases = [Path.cwd(), *Path.cwd().parents]
candidate_roots = []
for base in search_bases:
    candidate_roots.extend(
        [
            base,
            base / "simple_anomaly_detection",
            base / "python" / "simple_anomaly_detection",
        ]
    )
project_root = next((root for root in candidate_roots if (root / "notebook_support.py").exists()), None)
if project_root is None:
    raise FileNotFoundError("Could not locate the simple_anomaly_detection project root.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import notebook_support as ns
ns = importlib.reload(ns)

dataset_name_source = ns.RAW_DATASET_DIR if any(ns.RAW_DATASET_DIR.glob("*.txt")) else ns.LEGACY_VIRGIN_DIR
dataset_names = [path.stem for path in sorted(dataset_name_source.glob("*.txt"))]

panel_bundle = ns.build_control_panel(dataset_names)
NOTEBOOK_STATE = {
    "project_root": project_root,
    "ns": ns,
    "controls": panel_bundle["controls"],
    "panel_bundle": panel_bundle,
    "config": None,
    "context": None,
    "benchmark": None,
}

notes_info = ns.write_high_roi_algorithm_notes()
display(panel_bundle["panel"])
display(ns.build_results_layout_frame())
display(ns.list_paper_presets())
display(ns.build_algorithm_reference_overview())
print(f"Project root: {ns.PROJECT_ROOT}")
print(f"Legacy virgin source: {ns.LEGACY_VIRGIN_DIR}")
print(f"Results root: {ns.RESULTS_DIR}")
print(f"Results tables: {ns.RESULT_TABLES_DIR}")
print(f"Results figures: {ns.RESULT_FIGURES_DIR}")
print(f"Score traces: {ns.RESULT_SCORES_DIR}")
print(f"Algorithm notes: {notes_info['source_path']}")
print(f"Notes results copy: {notes_info['results_path']}")

,output_group,path,exists
0,notes/source,high_roi_algorithm_notes.md,True
1,notes/results_copy,results\tables\high_roi_algorithm_notes.md,True
2,results_root,results,True
3,tables,results\tables,True
4,tables/per_algorithm,results\tables\per_algorithm,True
5,figures,results\figures,True
6,figures/algorithm_panels,results\figures\algorithm_panels,True
7,figures/deep_dives,results\figures\deep_dives,True
8,figures/thesis,results\figures\thesis,True
9,scores,results\scores,True


,preset_name,label,enabled_algorithms,variant_count,description
0,auto_ablation,Auto Ablation Sweep,"Isolation Forest, Local Outlier Factor, SAND, ...",40,True one-factor-at-a-time ablation. Each enabl...
1,paper_full_suite,Paper Full Suite Sweep,"Isolation Forest, Local Outlier Factor, SAND, ...",24,Broader sweep across all implemented algorithm...
2,paper_high_roi,Paper High ROI Sweep,"Isolation Forest, Local Outlier Factor, Matrix...",12,Focused sweep for the strongest runtime/perfor...


,algorithm,implementation,summary,visible_controls,paper_high_roi_variants,paper_full_suite_variants
0,Isolation Forest,algorithms/isolation_forest.py,Fits an Isolation Forest on sliding windows an...,"Trees, Max samples, Max feat., Bootstrap, Seed",3,3
1,Local Outlier Factor,algorithms/local_outlier_factor.py,Computes LOF on sliding windows and uses the n...,"Neighbors, Search, Leaf size, Metric, p",3,3
2,SAND,algorithms/sand.py,Runs the legacy SAND online detector on a subs...,"Alpha, Init length, Batch size, k, Subseq x, O...",0,2
3,Matrix Profile,algorithms/matrix_profile.py,Uses the first column of `stumpy.stump` as a d...,Subseq x,3,3
4,DAMP,algorithms/damp.py,Runs the DAMP streaming-discord search and use...,"Start x, x_lag x",0,3
5,HBOS,algorithms/hbos.py,Builds one histogram per window position and s...,"Bins, Alpha, Tol",0,3
6,OCSVM,algorithms/ocsvm.py,Fits One-Class SVM on the earliest normalized ...,"Kernel, Nu, Gamma, Train frac",3,4
7,PCA,algorithms/pca.py,Fits PCA on sliding windows and scores each wi...,"Components, Score comps, Whiten, Weighted, Sta...",0,3


Project root: D:\Dev\Thesis\python\simple_anomaly_detection
Legacy virgin source: D:\Dev\Thesis\python\datasets\virgin
Results root: D:\Dev\Thesis\python\simple_anomaly_detection\results
Results tables: D:\Dev\Thesis\python\simple_anomaly_detection\results\tables
Results figures: D:\Dev\Thesis\python\simple_anomaly_detection\results\figures
Score traces: D:\Dev\Thesis\python\simple_anomaly_detection\results\scores
Algorithm notes: high_roi_algorithm_notes.md
Notes results copy: results\tables\high_roi_algorithm_notes.md


## Optional: Apply A Reproducible Automatic Experiment Layout

Edit `preset_name` below if you want a different automatic experiment layout. This also switches `Argument mode` to the selected auto sweep, then rerun the preparation and benchmark cells.

In [6]:
preset_name = "auto_ablation"
state = NOTEBOOK_STATE
state["ns"].apply_paper_experiment_preset(state["controls"], preset_name)
display(state["ns"].build_preset_reference_table(preset_name))
print(f"Applied preset: {preset_name}")

,preset_name,algorithm,variant_index,variant_name,focus,params_json
0,auto_ablation,Isolation Forest,1,Baseline,Stable tree baseline for comparison.,"{""ablation_label"": ""Baseline"", ""ablation_param..."
1,auto_ablation,Isolation Forest,2,Trees 400,Measures whether a larger forest improves scor...,"{""ablation_label"": ""Trees 400"", ""ablation_para..."
2,auto_ablation,Isolation Forest,3,Max samples auto,Measures how a broader training sample per tre...,"{""ablation_label"": ""Max samples auto"", ""ablati..."
3,auto_ablation,Isolation Forest,4,Max feat 0.6,Measures the effect of stronger random feature...,"{""ablation_label"": ""Max feat 0.6"", ""ablation_p..."
4,auto_ablation,Isolation Forest,5,Bootstrap on,Measures whether sampling windows with replace...,"{""ablation_label"": ""Bootstrap on"", ""ablation_p..."
...,...,...,...,...,...,...
35,auto_ablation,PCA,2,Components 0.95,Measures explained-variance truncation rather ...,"{""ablation_label"": ""Components 0.95"", ""ablatio..."
36,auto_ablation,PCA,3,Score comps 2,Measures scoring on a narrow set of trailing l...,"{""ablation_label"": ""Score comps 2"", ""ablation_..."
37,auto_ablation,PCA,4,Whiten on,Measures whitening of retained components befo...,"{""ablation_label"": ""Whiten on"", ""ablation_para..."
38,auto_ablation,PCA,5,Weighted off,Measures the score without explained-variance ...,"{""ablation_label"": ""Weighted off"", ""ablation_p..."


Applied preset: auto_ablation


## On Run: Read The Current Widget Values, Prepare Raw Datasets, And Build Normalized Benchmark Inputs

This next cell reads the current control-panel settings, ensures the raw datasets exist in the workspace, generates normalized labeled datasets if needed, and shows the exact paper run configuration.

In [7]:
state = NOTEBOOK_STATE
state["config"] = state["ns"].get_run_config(state["controls"])
state["context"] = state["ns"].prepare_run_context(state["config"])

display(state["context"]["run_config_frame"])
display(state["context"]["selected_run_parameters"])
display(state["context"]["preparation_summary"])
display(pd.DataFrame({"sample_raw_dataset_file": [path.name for path in state["context"]["raw_dataset_paths"][:10]]}))
print(f"Prepared datasets will be loaded from: {state['context']['prepared_dataset_dir']}")
print(f"Benchmark dataset count: {len(state['context']['benchmark_dataset_paths'])}")

,dataset_limit,batch_size,resume_from_existing,variant_mode,auto_preset_name,normalization_method,clip_quantile,window_size,window_stride,threshold_method,threshold_value,evaluation_mode,selected_algorithms,selected_configurations,variant_tabs,auto_filtered_out,selected_dataset_count,pending_dataset_count,planned_batch_count,next_batch_first_dataset,next_batch_last_dataset,prepared_dataset_dir,notes_source_path,notes_results_path
0,all,all selected,True,auto_ablation,auto_ablation,zscore,None,None,1,sigma,3.0,range,"isolation_forest, local_outlier_factor, sand, ...",40,"Isolation Forest: Baseline, Trees 400, Max sam...",None,241,241,241,053_UCR_Anomaly_DISTORTEDWalkingAceleration1_1...,240_UCR_Anomaly_taichidbS0715Master_240030_884...,datasets\normalized\zscore,high_roi_algorithm_notes.md,results\tables\high_roi_algorithm_notes.md


,dataset_limit,variant_mode,normalization_method,clip_quantile,window_size,window_stride,threshold_method,threshold_value,evaluation_mode,algorithm,algorithm_base_display,algorithm_display,algorithm_variant,algorithm_run_id,variant_index,variant_origin,variant_source,variant_focus,variant_family,ablation_parameter,ablation_label,ablation_role,params_json,param__n_estimators,param__max_samples,param__max_features,param__bootstrap,param__random_state,param__n_neighbors,param__algorithm,param__leaf_size,param__metric,param__p,param__alpha,param__init_length,param__batch_size,param__k,param__subsequence_multiplier,param__overlap,param__start_index_multiplier,param__x_lag_multiplier,param__n_bins,param__tol,param__kernel,param__nu,param__gamma,param__train_fraction,param__n_components,param__n_selected_components,param__whiten,param__weighted,param__standardization
0,all,auto_ablation,zscore,None,None,1,sigma,3.0,range,damp,DAMP,DAMP | Baseline,Baseline,damp__tab_01,1,auto,auto_ablation,Reference streaming-discord configuration.,baseline,baseline,Baseline,baseline,"{""start_index_multiplier"": 1.0, ""x_lag_multipl...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,all,auto_ablation,zscore,None,None,1,sigma,3.0,range,damp,DAMP,DAMP | Start x2,Start x2,damp__tab_02,2,auto,auto_ablation,Measures the effect of waiting longer before t...,ablation,start_index_multiplier,Start x2,score_driver,"{""start_index_multiplier"": 2.0, ""x_lag_multipl...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,all,auto_ablation,zscore,None,None,1,sigma,3.0,range,damp,DAMP,DAMP | x_lag x8,x_lag x8,damp__tab_03,3,auto,auto_ablation,Measures a much deeper historical search horiz...,ablation,x_lag_multiplier,x_lag x8,score_driver,"{""start_index_multiplier"": 1.0, ""x_lag_multipl...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,all,auto_ablation,zscore,None,None,1,sigma,3.0,range,hbos,HBOS,HBOS | Baseline,Baseline,hbos__tab_01,1,auto,auto_ablation,Lightweight histogram baseline.,baseline,baseline,Baseline,baseline,"{""alpha"": 0.1, ""n_bins"": 10, ""tol"": 0.5}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,all,auto_ablation,zscore,None,None,1,sigma,3.0,range,hbos,HBOS,HBOS | Bins 20,Bins 20,hbos__tab_02,2,auto,auto_ablation,Measures finer histogram resolution at every w...,ablation,n_bins,Bins 20,score_driver,"{""alpha"": 0.1, ""n_bins"": 20, ""tol"": 0.5}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35,all,auto_ablation,zscore,None,None,1,sigma,3.0,range,sand,SAND,SAND | Init 3000,Init 3000,sand__tab_03,3,auto,auto_ablation,Measures a shorter initialization phase before...,ablation,init_length,Init 3000,score_driver,"{""alpha"": 0.5, ""batch_size"": 2000, ""init_lengt...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,3000.0,2000.0,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,all,auto_ablation,zscore,None,None,1,sigma,3.0,range,sand,SAND,SAND | Batch 1000,Batch 1000,sand__tab_04,4,auto,auto_ablation,Measures finer-grained online updates at the c...,ablation,batch_size,Batch 1000,score_driver,"{""alpha"": 0.5, ""batch_size"": 1000, ""init_lengt...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,5000.0,1000.0,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37,all,auto_ablation,zscore,None,None,1,sigma,3.0,range,sand,SAND,SAND | k = 3,k = 3,sand__tab_05,5,auto,auto_ablation,Measures a more local nearest-subsequence comp...,ablation,k,k = 3,score_driver,"{""alpha"": 0.

,legacy_virgin_count,workspace_raw_count,workspace_normalized_count,benchmark_count,selected_dataset_count,batch_size,resume_from_existing,completed_dataset_count,pending_dataset_count,current_batch_count,next_batch_first_dataset,next_batch_last_dataset,variant_mode,auto_preset_name,normalization_method,clip_quantile,window_size,window_stride,threshold_method,threshold_value,evaluation_mode,normalized_dataset_dir,notes_source_path,notes_results_path
0,241,241,241,241,241,all selected,True,0,241,241,053_UCR_Anomaly_DISTORTEDWalkingAceleration1_1...,240_UCR_Anomaly_taichidbS0715Master_240030_884...,auto_ablation,auto_ablation,zscore,None,None,1,sigma,3.0,range,datasets\normalized\zscore,high_roi_algorithm_notes.md,results\tables\high_roi_algorithm_notes.md


,sample_raw_dataset_file
0,001_UCR_Anomaly_DISTORTED1sddb40_35000_52000_5...
1,002_UCR_Anomaly_DISTORTED2sddb40_35000_56600_5...
2,003_UCR_Anomaly_DISTORTED3sddb40_35000_46600_4...
3,004_UCR_Anomaly_DISTORTEDBIDMC1_2500_5400_5600...
4,005_UCR_Anomaly_DISTORTEDCIMIS44AirTemperature...
5,006_UCR_Anomaly_DISTORTEDCIMIS44AirTemperature...
6,007_UCR_Anomaly_DISTORTEDCIMIS44AirTemperature...
7,008_UCR_Anomaly_DISTORTEDCIMIS44AirTemperature...
8,009_UCR_Anomaly_DISTORTEDCIMIS44AirTemperature...
9,010_UCR_Anomaly_DISTORTEDCIMIS44AirTemperature...


Prepared datasets will be loaded from: D:\Dev\Thesis\python\simple_anomaly_detection\datasets\normalized\zscore
Benchmark dataset count: 241


## On Run: Execute Every Selected Algorithm Configuration Across The Prepared Datasets

This next cell refreshes the configuration from the widgets, runs every selected algorithm variant, writes the benchmark outputs into `results/tables/`, and stores all benchmark objects inside `NOTEBOOK_STATE["benchmark"]`.

In [8]:
state = NOTEBOOK_STATE
state["config"] = state["ns"].get_run_config(state["controls"])
state["context"] = state["ns"].prepare_run_context(state["config"])

try:
    state["benchmark"] = state["ns"].run_benchmark(
        state["config"],
        state["context"]["prepared_dataset_dir"],
        state["context"]["benchmark_dataset_paths"],
    )
except KeyboardInterrupt:
    print("Benchmark interrupted. Loading the latest partial results from the saved run session...")
    state = state["ns"].ensure_notebook_state_benchmark(
        state,
        refresh_from_saved_run=True,
        persist_tables=True,
        write_global=False,
    )
    print("Partial benchmark snapshot loaded into NOTEBOOK_STATE['benchmark'].")

display(state["benchmark"]["overview"])
display(state["benchmark"]["algorithm_summary"])
display(state["benchmark"]["dataset_catalog"].head(12))
if not state["benchmark"]["errors"].empty:
    display(state["benchmark"]["errors"][["dataset_name", "algorithm_display", "error"]])

print(f"Saved benchmark outputs to: {state['ns'].RESULT_TABLES_DIR}")


KeyboardInterrupt: 

## On Run: Explain How To Read The Ablation Outputs

This next cell is most useful when `Argument mode = auto_ablation`. It explains that each non-baseline variant changes one knob at a time, shows paired delta tables against the baseline on the same dataset, adds a short written summary of the strongest global gain/loss, and renders a high-level ablation overview figure with confidence intervals so you can defend which controls materially help, hurt, or mainly affect runtime.

In [ ]:
state = NOTEBOOK_STATE
state["ns"].render_ablation_overview(state)

## On Run: Show The Paper-Oriented Regime Summaries And Winners

This next cell displays the family summary, the regime-aware summary table, the best configuration by the selected evaluation metric, and the ROC-AUC winners table.

In [ ]:
state = NOTEBOOK_STATE
state = state["ns"].ensure_notebook_state_benchmark(
    state,
    refresh_from_saved_run=True,
    persist_tables=True,
    write_global=False,
)

display(state["benchmark"]["family_summary"])
display(state["benchmark"]["overall_regime_summary"])
display(state["benchmark"]["best_by_evaluation"].head(15))
display(state["benchmark"]["best_by_auc"].head(15))


## On Run: Show Isolation Forest Paper Report

This next cell renders the paper-facing report for Isolation Forest: a plain-language interpretation block, the configured variants, parameter-effect tables, regime-aware summaries by dataset type, the benchmark panel, the paper panel, and, when `Argument mode = auto_ablation`, paired baseline-vs-knob delta tables plus an ablation panel that shows effect size, confidence intervals, runtime tradeoff, and regime sensitivity. It also renders a fixed-layout side-by-side variant comparison graph on one shared dataset and the best showcase plot.

In [ ]:
state = NOTEBOOK_STATE
state["ns"].render_algorithm_report(state, "isolation_forest")

## On Run: Show Local Outlier Factor Paper Report

This next cell renders the paper-facing report for Local Outlier Factor: a plain-language interpretation block, the configured variants, parameter-effect tables, regime-aware summaries by dataset type, the benchmark panel, the paper panel, and, when `Argument mode = auto_ablation`, paired baseline-vs-knob delta tables plus an ablation panel that shows effect size, confidence intervals, runtime tradeoff, and regime sensitivity. It also renders a fixed-layout side-by-side variant comparison graph on one shared dataset and the best showcase plot.

In [ ]:
state = NOTEBOOK_STATE
state["ns"].render_algorithm_report(state, "local_outlier_factor")

## On Run: Show SAND Paper Report

This next cell renders the paper-facing report for SAND: a plain-language interpretation block, the configured variants, parameter-effect tables, regime-aware summaries by dataset type, the benchmark panel, the paper panel, and, when `Argument mode = auto_ablation`, paired baseline-vs-knob delta tables plus an ablation panel that shows effect size, confidence intervals, runtime tradeoff, and regime sensitivity. It also renders a fixed-layout side-by-side variant comparison graph on one shared dataset and the best showcase plot.

In [ ]:
state = NOTEBOOK_STATE
state["ns"].render_algorithm_report(state, "sand")

## On Run: Show Matrix Profile Paper Report

This next cell renders the paper-facing report for Matrix Profile: a plain-language interpretation block, the configured variants, parameter-effect tables, regime-aware summaries by dataset type, the benchmark panel, the paper panel, and, when `Argument mode = auto_ablation`, paired baseline-vs-knob delta tables plus an ablation panel that shows effect size, confidence intervals, runtime tradeoff, and regime sensitivity. It also renders a fixed-layout side-by-side variant comparison graph on one shared dataset and the best showcase plot.

In [ ]:
state = NOTEBOOK_STATE
state["ns"].render_algorithm_report(state, "matrix_profile")

## On Run: Show DAMP Paper Report

This next cell renders the paper-facing report for DAMP: a plain-language interpretation block, the configured variants, parameter-effect tables, regime-aware summaries by dataset type, the benchmark panel, the paper panel, and, when `Argument mode = auto_ablation`, paired baseline-vs-knob delta tables plus an ablation panel that shows effect size, confidence intervals, runtime tradeoff, and regime sensitivity. It also renders a fixed-layout side-by-side variant comparison graph on one shared dataset and the best showcase plot.

In [ ]:
state = NOTEBOOK_STATE
state["ns"].render_algorithm_report(state, "damp")

## On Run: Show HBOS Paper Report

This next cell renders the paper-facing report for HBOS: a plain-language interpretation block, the configured variants, parameter-effect tables, regime-aware summaries by dataset type, the benchmark panel, the paper panel, and, when `Argument mode = auto_ablation`, paired baseline-vs-knob delta tables plus an ablation panel that shows effect size, confidence intervals, runtime tradeoff, and regime sensitivity. It also renders a fixed-layout side-by-side variant comparison graph on one shared dataset and the best showcase plot.

In [ ]:
state = NOTEBOOK_STATE
state["ns"].render_algorithm_report(state, "hbos")

## On Run: Show OCSVM Paper Report

This next cell renders the paper-facing report for OCSVM: a plain-language interpretation block, the configured variants, parameter-effect tables, regime-aware summaries by dataset type, the benchmark panel, the paper panel, and, when `Argument mode = auto_ablation`, paired baseline-vs-knob delta tables plus an ablation panel that shows effect size, confidence intervals, runtime tradeoff, and regime sensitivity. It also renders a fixed-layout side-by-side variant comparison graph on one shared dataset and the best showcase plot.

In [ ]:
state = NOTEBOOK_STATE
state["ns"].render_algorithm_report(state, "ocsvm")

## On Run: Show PCA Paper Report

This next cell renders the paper-facing report for PCA: a plain-language interpretation block, the configured variants, parameter-effect tables, regime-aware summaries by dataset type, the benchmark panel, the paper panel, and, when `Argument mode = auto_ablation`, paired baseline-vs-knob delta tables plus an ablation panel that shows effect size, confidence intervals, runtime tradeoff, and regime sensitivity. It also renders a fixed-layout side-by-side variant comparison graph on one shared dataset and the best showcase plot.

In [ ]:
state = NOTEBOOK_STATE
state["ns"].render_algorithm_report(state, "pca")

## On Run: Build The Cross-Configuration Comparison Charts And Save The Final Overview Images

This next cell builds the overall comparison charts across all selected configurations by calling the reusable plotting helpers in `notebook_support.py`, saves the overview images into `results/figures/`, and displays the winner tables together with the regime-aware benchmark table.

In [ ]:
state = NOTEBOOK_STATE
state = state["ns"].ensure_notebook_state_benchmark(
    state,
    refresh_from_saved_run=True,
    persist_tables=True,
    write_global=False,
)
ns = state["ns"]
benchmark = state["benchmark"]
plt = ns._load_plotting_module()

figure_jobs = [
    ("benchmark_overview.png", lambda: ns.plot_benchmark_overview_panel(benchmark, ns.result_figure_path("benchmark_overview.png"))),
    ("pareto_frontier.png", lambda: ns.plot_pareto_frontier_panel(benchmark, ns.result_figure_path("pareto_frontier.png"))),
    ("metric_heatmap.png", lambda: ns.plot_metric_heatmap_panel(benchmark, ns.result_figure_path("metric_heatmap.png"))),
    ("family_evaluation_heatmap.png", lambda: ns.plot_family_evaluation_heatmap_panel(benchmark, ns.result_figure_path("family_evaluation_heatmap.png"))),
    ("algorithm_wins.png", lambda: ns.plot_algorithm_wins_panel(benchmark, ns.result_figure_path("algorithm_wins.png"))),
]

for filename, builder in figure_jobs:
    fig = builder()
    if fig is not None:
        plt.show()
        plt.close(fig)
        print(f"Saved: {ns.result_figure_path(filename)}")

display(benchmark["best_by_evaluation"].head(15))
display(benchmark["best_by_auc"].head(15))
display(benchmark["overall_regime_summary"])


## Optional: Export A Thesis-Ready Figure Pack

This next cell exports a fixed-style set of thesis-ready figures into `results/figures/thesis/` as both PNG and PDF, then writes a figure catalog plus ready-to-paste caption text into `results/tables/thesis_figure_catalog.csv` and `results/tables/thesis_figure_captions.md`.

In [ ]:
state = NOTEBOOK_STATE
thesis_export = state["ns"].export_thesis_figure_pack(state)
display(thesis_export["catalog"])
print(f"Saved thesis figures to: {thesis_export['figure_dir']}")
print(f"Saved figure catalog to: {thesis_export['catalog_path']}")
print(f"Saved captions to: {thesis_export['captions_path']}")